# Day 1 / Session 1: Modern LLM Foundations -- Exercises

These exercises reinforce the concepts from the demo notebook (`D1S1_demos.ipynb`).
Work through the **required exercises** (1--3) first, then tackle the **optional
exercises** if time allows.

| # | Exercise | Difficulty | Demos Referenced |
|---|----------|------------|------------------|
| 1 | Simple Conversational Agent | Easy (Required) | Demo 1, 4, 5 |
| 2 | Token Budget Calculator | Easy (Required) | Demo 2 |
| 3 | Parameter Advisor | Easy (Required) | Demo 3 |
| 4 | Multi-Persona Pipeline | Intermediate (Optional) | Demo 4, 5 |
| 5 | Self-Improving Response | Intermediate (Optional) | Demo 7 |
| 6 | Conversation with Token-Aware Summarization | Intermediate (Optional) | Demo 2, 4, 5 |

In [ ]:
!pip install -q openai tiktoken matplotlib scikit-learn python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv() # Load environment variables from .env

# ============================================================
# Imports and Setup
# ============================================================

import openai
import tiktoken
import os

print("All imports successful!")
print(f"OpenAI SDK version: {openai.__version__}")

---
## Recap

In the demos you saw how to initialize the OpenAI client, control model output with
parameters like `temperature` and `max_tokens`, shape behavior through system messages,
and chain multiple LLM calls into a pipeline. Now you will put those pieces together.

---
## Exercise 1: Build a Simple Conversational Agent (Easy -- Required)

You are building an engagement-manager assistant that a consulting team will use during
client engagements. Most of the code is provided -- you just need to fill in a few
key pieces.

The assistant should:
1. Maintain a **message history** list so it remembers prior turns.
2. Use a **system message** that defines it as a structured, data-driven consultant.
3. Accept user input, append it to the history, call the LLM, and append the reply.

> ** Key Concept:** The LLM sees the *entire* `messages` list on every call.
> That is how it "remembers" earlier turns -- it re-reads them each time.

**References:** Demo 1 (basic API call), Demo 4 (personas), Demo 5 (pipeline helper)

In [ ]:
# Exercise 1: Build a Simple Conversational Agent
#
# Most of the code is provided. You only need to fill in the lines marked TODO.

class SimpleAgent:
    def __init__(self):
        """Initialize the agent with a consulting persona."""
        self.client = openai.OpenAI()
        self.model = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")

        # The system message tells the LLM what role to play.
        # This message stays at the start of the history list forever.
        self.history = [
            {"role": "system", "content": "You are a McKinsey engagement manager assistant. You help consulting teams with structured, data-driven analysis using MECE frameworks. Maintain a professional tone and provide actionable recommendations."}
        ]

    def chat(self, user_input):
        """Process a user message and return the assistant's response."""

        # TODO 1: Append the user's message to self.history
        #         Format: {"role": "user", "content": user_input}
        #  Hint: self.history.append({"role": "...", "content": ...})


        # Call the LLM with the FULL history (so it sees all prior turns)
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.history,                                   # full history
            temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")),
            max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
        )

        # TODO 2: Extract the reply text from the response object
        #          Hint: reply = response.choices[0].message.content
        reply = None   # <-- replace None with the correct expression


        # TODO 3: Append the assistant's reply to self.history
        #         Format: {"role": "assistant", "content": reply}


        return reply

    def get_history(self):
        """Return the full conversation history."""
        return self.history

### Test Your Agent

Uncomment and run the cell below once you have filled in the class.

In [ ]:
# Uncomment to test your agent with a multi-turn conversation

# agent = SimpleAgent()
#
# # Turn 1
# print("USER: We have a new client in the automotive sector facing margin pressure. What should we focus on?")
# print(f"AGENT: {agent.chat('We have a new client in the automotive sector facing margin pressure. What should we focus on?')}")
# print()
#
# # Turn 2 -- the agent should remember the automotive context
# print("USER: Which of those areas would have the quickest impact?")
# print(f"AGENT: {agent.chat('Which of those areas would have the quickest impact?')}")
# print()
#
# # Verify history
# history = agent.get_history()
# print(f"Conversation history: {len(history)} messages")
# for msg in history:
# print(f" [{msg['role']}] {msg['content'][:80]}{'...' if len(msg['content']) > 80 else ''}")

### Expected Output

A working solution will produce output similar to the following (exact wording will
vary because the LLM is non-deterministic):

```
USER: We have a new client in the automotive sector facing margin pressure. What should we focus on?
AGENT: To address margin pressure in the automotive sector, I would recommend structuring
 our analysis around these MECE workstreams:
 1. Revenue levers -- pricing strategy, product mix, geographic expansion
 2. Cost structure -- procurement, manufacturing efficiency, SG&A optimization
 3. Portfolio strategy -- divesting low-margin segments, investing in high-growth areas
 4. Operational excellence -- lean operations, supply chain resilience

USER: Which of those areas would have the quickest impact?
AGENT: For the automotive client you described, cost structure optimization -- particularly
 procurement and manufacturing efficiency -- typically yields the quickest results.
 These levers can often be activated within 3-6 months...

Conversation history: 5 messages
 [system] You are a McKinsey engagement manager assistant. You help...
 [user] We have a new client in the automotive sector facing margin pressure...
 [assistant] To address margin pressure in the automotive sector, I would rec...
 [user] Which of those areas would have the quickest impact?...
 [assistant] For the automotive client you described, cost structure optimizat...
```

**Key check:** The agent must remember context from Turn 1 (automotive, margin pressure)
when answering Turn 2. If it does not, the message history is not being passed correctly.

### Hints (try without looking first!)

<details>
<summary><strong>Hint 1:</strong> TODO 1 -- appending the user message</summary>

```python
self.history.append({"role": "user", "content": user_input})
```
This is just a normal Python `list.append()`. The dict must have `"role"` and `"content"` keys.
</details>

<details>
<summary><strong>Hint 2:</strong> TODO 2 -- extracting the reply</summary>

```python
reply = response.choices[0].message.content
```
The API always returns a list of `choices`. We take the first one and read its `.message.content`.
</details>

<details>
<summary><strong>Hint 3:</strong> TODO 3 -- appending the assistant reply</summary>

```python
self.history.append({"role": "assistant", "content": reply})
```
Same pattern as TODO 1, but with `"role": "assistant"`.
</details>

<details>
<summary><strong>Hint 4:</strong> Why does Turn 2 know about automotive?</summary>

Because `self.history` is passed to every API call. By Turn 2, it contains:
`[system_msg, user_turn1, assistant_turn1, user_turn2]`

The model sees the entire conversation and can reference prior context.
</details>

---
## Exercise 2: Token Budget Calculator (Easy -- Required)

Write a function that checks whether a prompt fits within a model's context window.
Most of the code is provided -- you just need to fill in a few calculations.

> ** Key Concept:** Every model has a fixed context window (e.g., 128 000 tokens).
> Your prompt tokens + response tokens must fit inside this window. `tiktoken`
> lets you count tokens *before* sending a request.

**Reference:** Demo 2 for the `tiktoken` encoding pattern.

In [ ]:
# Exercise 2: Token Budget Calculator
#
# Fill in the lines marked TODO. The encoder setup is done for you.

import tiktoken

def check_token_budget(prompt, max_context=128000, reserved_for_response=500):
    """
    Check whether a prompt fits within the context window budget.

    Args:
        prompt: The text to tokenize and check
        max_context: Total context window size (default: 128000 for gpt-4o-mini)
        reserved_for_response: Tokens reserved for the model's response

    Returns:
        dict with keys: token_count, remaining_budget, fits, utilization_percentage
    """
    # Encoder setup (done for you)
    encoder = tiktoken.encoding_for_model("gpt-4o-mini")
    tokens = encoder.encode(prompt)
    token_count = len(tokens)

    # TODO 1: Calculate the available budget (max_context minus reserved_for_response)
    #  Hint: available = max_context - reserved_for_response
    available = None   # <-- replace None


    # TODO 2: Calculate remaining budget (available minus token_count)
    remaining = None   # <-- replace None


    # TODO 3: Does the prompt fit? (remaining should be >= 0)
    fits = None   # <-- replace None


    # TODO 4: Calculate utilization percentage: (token_count / available) * 100
    utilization = None   # <-- replace None


    return {
        "token_count": token_count,
        "remaining_budget": remaining,
        "fits": fits,
        "utilization_percentage": utilization
    }

### Test Your Token Budget Calculator

Uncomment and run the cell below once you have filled in the function.

In [ ]:
# Uncomment to test

# # Test 1: Short prompt
# result1 = check_token_budget("What is digital transformation?")
# print("Test 1 - Short prompt:")
# print(f" Token count: {result1['token_count']}")
# print(f" Remaining budget: {result1['remaining_budget']}")
# print(f" Fits: {result1['fits']}")
# print(f" Utilization: {result1['utilization_percentage']:.4f}%")
# print()
#
# # Test 2: Simulated long document
# long_doc = "The target company operates in a fragmented market. " * 5000
# result2 = check_token_budget(long_doc)
# print("Test 2 - Long document (simulated due diligence):")
# print(f" Token count: {result2['token_count']}")
# print(f" Remaining budget: {result2['remaining_budget']}")
# print(f" Fits: {result2['fits']}")
# print(f" Utilization: {result2['utilization_percentage']:.2f}%")
# print()
#
# # Test 3: Smaller context window (simulating gpt-3.5-turbo)
# result3 = check_token_budget(long_doc, max_context=16385)
# print("Test 3 - Same document, smaller context window (16k):")
# print(f" Token count: {result3['token_count']}")
# print(f" Remaining budget: {result3['remaining_budget']}")
# print(f" Fits: {result3['fits']}")
# print(f" Utilization: {result3['utilization_percentage']:.2f}%")

### Expected Output

```
Test 1 - Short prompt:
 Token count: 5
 Remaining budget: 127495
 Fits: True
 Utilization: 0.0039%

Test 2 - Long document (simulated due diligence):
 Token count: ~45000
 Remaining budget: ~82500
 Fits: True
 Utilization: ~35%

Test 3 - Same document, smaller context window (16k):
 Token count: ~45000
 Remaining budget: ~-29115
 Fits: False
 Utilization: ~283%
```

(Exact token counts may vary slightly depending on tiktoken version.)

### Hints

<details>
<summary><strong>Hint 1:</strong> TODO 1 & 2 -- the math</summary>

```python
available = max_context - reserved_for_response   # e.g. 128000 - 500 = 127500
remaining = available - token_count                # how many tokens you have left
```
</details>

<details>
<summary><strong>Hint 2:</strong> TODO 3 & 4</summary>

```python
fits = remaining >= 0                              # True if prompt fits
utilization = (token_count / available) * 100      # percentage of budget used
```
</details>

---
## Exercise 3: Parameter Advisor (Easy -- Required)

Write a function that recommends LLM parameters for different consulting task types,
then test it by actually calling the LLM with those parameters.

Two presets are provided for you. You need to add the remaining two and handle
the unknown-task default.

> ** Key Concept:** Low temperature -> deterministic, focused output.
> High temperature -> creative, varied output. Match the parameters to the task!

**Reference:** Demo 3 for how `temperature`, `top_p`, and `max_tokens` affect output.

In [ ]:
# Exercise 3: Parameter Advisor
#
# Two presets are done for you. Add the remaining two + a default.

def suggest_parameters(task_type):
    """
    Suggest LLM parameters based on the type of consulting task.

    Args:
        task_type: One of "financial_report", "brainstorming",
                   "classification", "creative_writing"

    Returns:
        dict with keys: temperature, top_p, max_tokens, reasoning
    """
    presets = {
        # [OK] Provided: financial reports need precision -> low temperature
        "financial_report": {
            "temperature": 0.1,
            "top_p": 0.3,
            "max_tokens": 300,
            "reasoning": "Financial reports require deterministic, precise output with minimal variation."
        },
        # [OK] Provided: brainstorming needs creativity -> high temperature
        "brainstorming": {
            "temperature": 0.9,
            "top_p": 0.95,
            "max_tokens": 500,
            "reasoning": "Brainstorming benefits from high creativity and diverse ideas."
        },

        # TODO 1: Add "classification" preset
        #         Classification needs very consistent labels -> very low temp
        #          Hint: temperature=0.0, top_p=0.1, max_tokens=50


        # TODO 2: Add "creative_writing" preset
        #         Creative writing needs variety -> high temp, long output
        #          Hint: temperature=0.8, top_p=0.9, max_tokens=800

    }

    # TODO 3: Return the matching preset, or a default if task_type is unknown
    #  Hint: use presets.get(task_type, {default_dict})
    return presets.get(task_type, {
        "temperature": 0.5,
        "top_p": 0.5,
        "max_tokens": 200,
        "reasoning": "Unknown task type, using defaults"
    })

### Test Your Parameter Advisor

Uncomment and run the cells below once you have filled in the function.

In [ ]:
# Uncomment to test the parameter suggestions

# task_types = ["financial_report", "brainstorming", "classification", "creative_writing", "unknown_task"]
#
# for task in task_types:
# params = suggest_parameters(task)
# print(f"Task: {task}")
# print(f" temperature : {params['temperature']}")
# print(f" top_p : {params['top_p']}")
# print(f" max_tokens : {params['max_tokens']}")
# print(f" reasoning : {params['reasoning']}")
# print()

In [ ]:
# Uncomment to test by actually calling the LLM with suggested parameters

# client = openai.OpenAI()
# test_prompt = "Describe the impact of AI on management consulting."
#
# for task in ["financial_report", "brainstorming"]:
# params = suggest_parameters(task)
# print(f"--- Task type: {task} ---")
# print(f"Parameters: temp={params['temperature']}, top_p={params['top_p']}, max_tokens={params['max_tokens']}")
# print(f"Reasoning: {params['reasoning']}")
# print()
#
# response = client.chat.completions.create(
# model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
# messages=[{"role": "user", "content": test_prompt}],
# temperature=params["temperature"],
# top_p=params["top_p"],
# max_tokens=params["max_tokens"]
# )
# print(f"Response (finish_reason={response.choices[0].finish_reason}):")
# print(f" {response.choices[0].message.content}")
# print()

### Expected Output

```
Task: financial_report
 temperature : 0.1
 top_p : 0.3
 max_tokens : 300
 reasoning : Financial reports require deterministic, precise output...

Task: brainstorming
 temperature : 0.9
 top_p : 0.95
 max_tokens : 500
 reasoning : Brainstorming benefits from high creativity and diverse ideas...

Task: classification
 temperature : 0.0
 top_p : 0.1
 max_tokens : 50
 reasoning : Classification needs consistent, deterministic labels...

Task: creative_writing
 temperature : 0.8
 top_p : 0.9
 max_tokens : 800
 reasoning : Creative writing needs variety and expressiveness...

Task: unknown_task
 temperature : 0.5
 top_p : 0.5
 max_tokens : 200
 reasoning : Unknown task type, using defaults
```

When you test with actual LLM calls, compare the `financial_report` output (focused, precise) with the `brainstorming` output (varied, creative). The difference should be visible.

### Hints

<details>
<summary><strong>Hint 1:</strong> Structure your parameter mappings</summary>

```python
presets = {
 "financial_report": {
 "temperature": 0.1,
 "top_p": 0.3,
 "max_tokens": 300,
 "reasoning": "Financial reports require ..."
 },
 # ... more task types
}
```
</details>

<details>
<summary><strong>Hint 2:</strong> Handling unknown task types</summary>

```python
return presets.get(task_type, {
 "temperature": 0.5,
 "top_p": 0.5,
 "max_tokens": 200,
 "reasoning": "Unknown task type, using defaults"
})
```
</details>

---
---
# Optional Exercises (Intermediate)

The exercises below are optional and at an intermediate level. They combine
concepts from multiple demos. Attempt them if you finish the core exercises early.

---
## Optional Exercise 1 (Intermediate): Multi-Persona Pipeline

Build a pipeline that sends the same client question to 3 different personas
(like Demo 4), collects their responses, then sends all 3 to a "synthesis agent"
that combines the best insights into a unified recommendation.

> ** Hint:** The overall flow is: define personas -> loop through each -> collect
> responses -> build a combined prompt -> call the LLM once more to synthesize.

**References:**
- **Demo 4** for persona system messages
- **Demo 5** for the `call_llm` helper and pipeline chaining pattern

In [ ]:
# Optional Exercise 1: Multi-Persona Pipeline
#
# TODO 1: Initialize the OpenAI client
#
# TODO 2: Define a call_llm helper function (same pattern as Demo 5)
#
# TODO 3: Define at least 3 personas as a dictionary mapping
# persona_name -> system_message (reuse or adapt from Demo 4)
#
# TODO 4: Define the client question
#
# TODO 5: Loop through each persona, call the LLM, and store responses
# in a dictionary: responses[persona_name] = response_text
#
# TODO 6: Build a synthesis prompt that includes all persona responses.
# Something like:
# "Here are analyses from three different McKinsey experts:
# --- Strategy Partner ---
# {responses['Strategy']}
# --- Operations Associate ---
# {responses['Operations']}
# --- Digital Expert ---
# {responses['Digital']}
# Synthesize these into a unified recommendation."
#
# TODO 7: Call the LLM with a synthesis system message and the combined prompt
#
# TODO 8: Print the synthesized result

client = openai.OpenAI()

def call_llm(system_message, user_message, temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")), max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))):
 """Helper function to make an LLM call."""
 # YOUR CODE HERE (copy from Demo 5 or rewrite)
 pass

# Define personas (adapt from Demo 4)
personas = {
 # YOUR CODE HERE
}

client_question = "Our telecom client is losing enterprise customers to smaller competitors. What should we do?"

# Collect responses from each persona
responses = {}
# YOUR CODE HERE -- loop through personas, collect responses

# Synthesize
# YOUR CODE HERE -- build synthesis prompt, call LLM, print result

---
## Optional Exercise 2 (Intermediate): Self-Improving Response

Build a reusable function that generates a response, then critiques it, then generates
an improved version -- like Demo 7's self-reflection capability, but packaged as a
reusable tool.

> ** Hint:** This is a 3-step chain: generate -> critique -> improve. Each step
> is a separate LLM call with a different system message.

**Reference:** Refer to **Demo 7, Capability 3** (self-reflection) for the pattern.

In [ ]:
# Optional Exercise 2: Self-Improving Response
#
# TODO 1: Initialize the OpenAI client
#
# TODO 2: In generate_initial(), call the LLM with a basic system message
# and the user's question. Return the response text.
#
# TODO 3: In critique(), send the original question AND the initial response
# to the LLM with a "critic" system message that evaluates for
# completeness, MECE structure, and analytical rigor.
# Return the critique text.
#
# TODO 4: In improve(), send the original question, initial response, AND
# the critique to the LLM with an "improver" system message that
# produces a better version incorporating the feedback.
# Return the improved text.
#
# TODO 5: In self_improving_response(), chain all three steps together
# and return a dict with all three outputs.

client = openai.OpenAI()

def self_improving_response(question):
 """
 Generate a response, critique it, and produce an improved version.
 
 Args:
 question: The user's question
 
 Returns:
 dict with keys: initial, critique, improved
 """
 # Step 1: Generate initial response
 # YOUR CODE HERE
 initial = None
 
 # Step 2: Critique the initial response
 # YOUR CODE HERE
 critique = None
 
 # Step 3: Generate improved response
 # YOUR CODE HERE
 improved = None
 
 return {
 "initial": initial,
 "critique": critique,
 "improved": improved
 }

# Uncomment to test:
# result = self_improving_response("What are the key success factors for a post-merger integration?")
# print("=== INITIAL RESPONSE ===")
# print(result["initial"])
# print("\n=== CRITIQUE ===")
# print(result["critique"])
# print("\n=== IMPROVED RESPONSE ===")
# print(result["improved"])

---
## Optional Exercise 3 (Intermediate): Conversation with Token-Aware Summarization

> **Note:** This exercise combines concepts from multiple demos and exercises.
> It ties together the conversational agent (Exercise 1), token counting
> (Exercise 2), and pipeline chaining (Demo 5).

Build a chat agent that:
1. Maintains a conversation history (like Exercise 1)
2. Tracks token usage of the full history after every turn
3. When the conversation history exceeds **70%** of the context window, automatically
   summarizes older messages to compress the history, keeping only the summary +
   recent messages

> ** Hint:** The compression step takes all messages except the system message
> and the last 2, sends them to the LLM for summarization, then replaces them
> with a single summary message.

This is a real-world pattern used in production chatbots to handle long conversations
without exceeding context limits.

In [ ]:
# Optional Exercise 3: Conversation with Token-Aware Summarization
#
# TODO 1: In __init__, set up client, model, encoder, history (with system message),
# max_context, reserved_for_response, and a compression_threshold (0.70)
#
# TODO 2: Implement _count_history_tokens() that serializes self.history to a
# string and counts tokens. (Hint: json.dumps or concatenate content strings)
#
# TODO 3: Implement _compress_history() that:
# a. Keeps the system message (history[0])
# b. Takes all messages except the system message and the last 2 messages
# c. Sends those older messages to the LLM with a prompt like:
# "Summarize this conversation concisely, preserving key context and decisions."
# d. Replaces the older messages with a single assistant message containing
# the summary, prefixed with "[CONVERSATION SUMMARY]: "
# e. Reconstructs history as: [system_msg, summary_msg, last_2_messages...]
#
# TODO 4: In chat(), after appending the assistant reply, check token usage.
# If utilization > compression_threshold, call _compress_history()
#
# TODO 5: In chat(), print token usage stats after each turn for visibility

import json
import tiktoken

class TokenAwareAgent:
 def __init__(self, max_context=128000, reserved_for_response=500):
 """Initialize the token-aware conversational agent."""
 # YOUR CODE HERE
 pass
 
 def _count_history_tokens(self):
 """Count the total tokens in the current conversation history."""
 # YOUR CODE HERE
 pass
 
 def _compress_history(self):
 """Summarize older messages to reduce token usage."""
 # YOUR CODE HERE
 pass
 
 def chat(self, user_input):
 """Process a user message, manage token budget, return response."""
 # YOUR CODE HERE
 pass
 
 def get_history(self):
 """Return the full conversation history."""
 # YOUR CODE HERE
 pass
 
 def get_token_usage(self):
 """Return current token usage statistics."""
 # YOUR CODE HERE
 pass

# Uncomment to test:
# agent = TokenAwareAgent(max_context=4000, reserved_for_response=500) # Small window to trigger compression
#
# questions = [
# "What are the top 3 trends in the pharmaceutical industry?",
# "How does each trend affect M&A activity in the sector?",
# "Which of these trends creates the biggest opportunity for private equity?",
# "Draft a one-page investment thesis for a PE fund focusing on this opportunity.",
# "What are the key risks to this thesis?",
# "How would you structure a 90-day diligence plan?",
# ]
#
# for i, q in enumerate(questions, 1):
# print(f"\n{'='*60}")
# print(f"Turn {i}: {q}")
# print(f"{'='*60}")
# reply = agent.chat(q)
# print(f"Agent: {reply[:200]}..." if len(reply) > 200 else f"Agent: {reply}")
# usage = agent.get_token_usage()
# print(f"\n[Token usage: {usage}]")

### What to Look For

When running with a small `max_context` (e.g., 4000), you should see:
1. Token usage climbing with each turn
2. At some point, the agent triggers compression and token count drops
3. Despite compression, the agent still remembers the overall conversation context
 (because the summary preserves key decisions)
4. The conversation can continue indefinitely without hitting the context limit

This is the same pattern used by ChatGPT and other production chatbots to handle
arbitrarily long conversations.